In [3]:
import pandas as pd

In [6]:
df = pd.read_csv('data/cleaned_online_retail.csv')

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 407695 entries, 0 to 407694
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      407695 non-null  int64  
 1   StockCode    407695 non-null  str    
 2   Description  407695 non-null  str    
 3   Quantity     407695 non-null  int64  
 4   InvoiceDate  407695 non-null  str    
 5   Price        407695 non-null  float64
 6   Customer ID  407695 non-null  float64
 7   Country      407695 non-null  str    
 8   Revenue      407695 non-null  float64
dtypes: float64(3), int64(2), str(4)
memory usage: 28.0 MB


In [8]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [9]:
analysis_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
analysis_date

Timestamp('2010-12-10 20:01:00')

In [10]:
rfm = df.groupby("Customer ID").agg({
    "InvoiceDate" : lambda x : (analysis_date - x.max()).days, 
    "Invoice" : "nunique", 
    "Revenue": "sum" 
    })
rfm.columns = ["Recency", "Frequency", "Monetary"]

In [11]:
rfm

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,165,11,372.86
12347.0,3,2,1323.32
12348.0,74,1,222.16
12349.0,43,3,2671.14
12351.0,11,1,300.93
...,...,...,...
18283.0,18,6,641.77
18284.0,67,1,461.68
18285.0,296,1,427.00


In [12]:
r_quintiles = pd.qcut(rfm["Recency"], 5, labels=[5,4,3,2,1])
f_quintiles = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1,2,3,4,5])
m_quintiles = pd.qcut(rfm["Monetary"], 5, labels = [1,2,3,4,5])
rfm["R"] = r_quintiles.astype(int)
rfm["F"] = f_quintiles.astype(int)
rfm["M"] = m_quintiles.astype(int)
rfm["RMF_score"] = rfm[["R", "F", "M"]].sum(axis=1)

In [13]:
rfm.head(5)

,Recency,Frequency,Monetary,R,F,M,RMF_score
Customer ID,,,,,,,
12346.0,165,11,372.86,2,5,2,9
12347.0,3,2,1323.32,5,2,4,11
12348.0,74,1,222.16,2,1,1,4
12349.0,43,3,2671.14,3,3,5,11
12351.0,11,1,300.93,5,1,2,8


In [14]:
def function(row): 
    if row["R"] >= 4 and row["F"] >= 4 and row["M"] >= 4:
        return "Baleine"
    elif row["R"] >= 4 and row["F"] < 3:
        return "Nouveau client"
    elif row["R"] >= 3 and row["F"] >= 4 and row["M"] >= 3:
        return "Client loyal"
    elif row["R"] >= 3 and row["F"] < 3  and row["M"] >= 3:
        return "Client prometteur"
    elif row["R"] < 2 and row["F"] >= 2 and row["M"] < 3:
        return "Client dormant"
    elif row["R"] < 3 and row["F"] >= 3 and row["M"] >= 4:
        return "Client a risque"
    elif  row["R"] < 3 and row["M"] >= 4 :
        return "Bon client perdu"
    else : 
        return "Autres clients"

rfm["type_client"] =  rfm.apply(function, axis=1)

In [ ]:
rfm.head(10)